补充学习一下Invoke()的返回值，仔细琢磨其各个字段的含义

In [4]:
from dotenv import load_dotenv
import os
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from rich import print as rich_print
load_dotenv()
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
    model="gpt-5.4-mini",
)

invoke_result = model.invoke([HumanMessage(content="你好，介绍一下你自己。")])
print(type(invoke_result))
rich_print(invoke_result)


<class 'langchain_core.messages.ai.AIMessage'>


AIMessage(
    content='你好！我是 ChatGPT，一个由 OpenAI 训练的人工智能助手。\n\n我可以帮你做很多事，比如：\n- 
回答问题、解释概念\n- 写作和润色文本\n- 翻译中英文或其他语言\n- 总结文章、提炼要点\n- 写代码、改代码、排查错误\n- 
帮你头脑风暴、做计划、整理思路\n\n我的特点是：\n- 可以快速处理大量信息\n- 能按你的需求调整表达风格\n- 
适合做学习、工作和日常咨询的辅助工具\n\n不过也有一些限制：\n- 
我不一定总是完全正确，尤其是涉及最新信息或复杂专业判断时\n- 我没有人的真实经历和情感\n- 
我不能直接访问你的私人数据，除非你提供给我\n\n如果你愿意，我也可以进一步用“更正式”“更亲切”或者“更像朋友”的方式重新
介绍自己。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 210,
            'prompt_tokens': 13,
            'total_tokens': 223,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': 0,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': None,
                'image_tokens': 0
            },
            'prompt_tokens_details': {
                'audio_tokens': 0,
                'cached_tokens': 0,
                'cache_write_tokens': 0,
                'video_tokens': 0
            },
            'cost': 0.00095475,
            'is_byok': False,
            'cost_details': {
                'upstream_inference_cost': 0.00095475,
                'upstream_inference_prompt_cost': 9.75e-06,
                'upstream_inference_completions_cost': 0.000945
            }
        },
        'model_provider': 'openai',
        'model_name': 'openai/gpt-5.4-mini-20260317',
        'system_fingerprint': None,
        'id': 'gen-1782526516-JKZrGHpX6ijAuGhMLBk9',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f06dc-1190-7291-8476-1238aa00345d-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 13,
        'output_tokens': 210,
        'total_tokens': 223,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 0}
    }
)

## `invoke()` 返回值解读：`AIMessage`

`model.invoke("你好")` 返回的不是普通字符串，而是一个 `langchain_core.messages.ai.AIMessage` 对象。它把"模型回的话"和"这次调用的元信息"打包在一起。下面按字段拆解。

### 1. `content` —— 模型回复的正文

```python
content = '你好！有什么我可以帮你的吗？'
```

这是你最常取的字段：`result.content`。多模态场景下它可能是 list（含文本/图片/工具调用片段），纯文本时就是 str。

### 2. `additional_kwargs` —— 厂商扩展字段（少用）

```python
{'refusal': None}  # OpenAI 特有：模型拒绝回答时这里会有原因，None 表示正常回答
```

存放 provider 特有、又不足以放进标准字段的信息。`refusal` 是 OpenAI 的"拒答原因"，这里为 `None` 说明回答正常。

### 3. `response_metadata` —— 本次请求的"回执"（重点）

这是最值得关注的字段，记录了这次调用本身的全部信息。

**① `token_usage` / Token 用量与费用**

| 字段 | 值 | 含义 |
|---|---|---|
| `prompt_tokens` | 7 | 输入 token 数（你发的"你好"被切成 7 个 token） |
| `completion_tokens` | 13 | 输出 token 数（模型回的那句话） |
| `total_tokens` | 20 | 上面两者之和 |
| `cost` | 6.375e-05 | 本次调用费用（美元），约 $0.0000638 |
| `is_byok` | False | 是否使用了自带密钥（Bring Your Own Key） |

**② `_details` 子结构 —— Token 的细分类**

- `completion_tokens_details`：输出 token 拆分
  - `reasoning_tokens: 0`：推理模型（o 系列）专属，思考用了多少 token。这里是 0，说明 gpt-5.4-mini 不走显式推理。
  - `audio_tokens / image_tokens: 0`：音频/图像输出 token。
  - `accepted/rejected_prediction_tokens: None`：预测缓存命中/被拒。
- `prompt_tokens_details`：输入 token 拆分
  - `cached_tokens: 0` / `cache_write_tokens: 0`：是否命中 OpenAI 的 prompt 缓存（命中会便宜很多）。
  - `audio_tokens / video_tokens: 0`：多模态输入的音视频 token。

**③ `cost_details` —— 费用拆分**

```python
'upstream_inference_prompt_cost':       5.25e-06   # 输入花费
'upstream_inference_completions_cost':  5.85e-05   # 输出花费
'upstream_inference_cost':              6.375e-05  # 合计
```
可以看出输出 token 单价明显高于输入（13 个输出 ≈ 7 个输入的 11 倍花费）。

**④ 模型与请求标识**

| 字段 | 值 | 含义 |
|---|---|---|
| `model_provider` | `'openai'` | 实际提供方 |
| `model_name` | `'openai/gpt-5.4-mini-20260317'` | 实际模型（带日期快照，保证可复现） |
| `id` | `'gen-1782525941-...'` | 服务端请求 ID（排查问题给客服用的就是它） |
| `service_tier` | `'default'` | 服务等级（default / priority / scale 等） |
| `system_fingerprint` | `None` | 模型权重指纹（OpenAI 用来标识后端版本，这里没返回） |

**⑤ `finish_reason: 'stop'` —— 结束原因（重要）**

告诉你是为什么结束的：
- `'stop'`：自然结束（最常见，本次就是）
- `'length'`：撞上 max_tokens 被截断
- `'tool_calls'`：模型要调用工具
- `'content_filter'`：被安全过滤拦截

**⑥ `logprobs: None`** —— 对数概率（调试用，需要单独开启才会返回）。

### 4. `id` —— LangChain 端的消息 ID

```python
id = 'lc_run--019f06d3-4ec8-7650-80ae-1d970defc72c-0'
```
注意这跟 `response_metadata['id']` 不是一回事：
- 外层 `id`（`lc_run--...`）：**LangChain 自己生成**的运行追踪 ID，用于 LangSmith 追踪。
- `response_metadata['id']`（`gen-...`）：**模型服务商**返回的请求 ID。

### 5. `tool_calls` / `invalid_tool_calls` —— 工具调用

```python
tool_calls = []          # 模型本次没要求调用任何工具
invalid_tool_calls = []  # 也没有解析失败的工具调用
```

当模型决定调用工具（function calling）时，这里会填充 `[{name, args, id}]`。本次是普通对话，所以为空。绑定工具后这里才会有内容。

### 6. `usage_metadata` —— 标准化用量（跨厂商通用）

```python
{'input_tokens': 7, 'output_tokens': 13, 'total_tokens': 20, ...}
```

这是 `response_metadata['token_usage']` 的**精简标准版**，字段名跨厂商统一（OpenAI、Anthropic、Google 都一样）。写代码统计 token 时优先用这个，别去翻 `response_metadata`。

### 7. ⚠️ 为什么我的返回里没有 `latency_checkpoint`？（厂商私有字段会被丢弃）

你在别的教程里可能见过 `response_metadata` 里有这样一坨延迟统计：

```python
'latency_checkpoint': {
    'engine_tbt_ms': 4,           # 引擎层 token 间隔
    'engine_ttft_ms': 36,         # 引擎层首 token 延迟 (Time To First Token)
    'engine_ttlt_ms': 100,        # 引擎层末 token 延迟 (Time To Last Token)
    'pre_inference_ms': 86,       # 推理前处理耗时
    'service_tbt_ms': 4,          # 服务层（含网络/排队）token 间隔
    'service_ttft_ms': 280,       # 服务层首 token 延迟
    'service_ttlt_ms': 338,       # 服务层末 token 延迟
    'total_duration_ms': 259,     # 总耗时
    'user_visible_ttft_ms': 194   # 用户实际感知的首 token 延迟
}
```

**这是 OpenRouter 私有字段**，不是 OpenAI 官方规范的一部分。本 notebook 里之所以看不到它，是因为代码用了 `init_chat_model(base_url=openrouter_base_url, ...)` —— 等同于用 **`ChatOpenAI`** 去打 OpenRouter。

LangChain 官方文档明确警告：

> `ChatOpenAI` 只解析 OpenAI 官方 API 规范，**第三方厂商/代理的非标准字段不会被提取或保留**。
>
> - `latency_checkpoint`（OpenRouter 延迟统计）
> - `reasoning_content` / `reasoning` / `reasoning_details`（DeepSeek/Qwen 的推理过程）
> 等都会被**静默丢弃**。

**想要拿到这些字段**，应改用厂商专属集成包，而不是裸 `ChatOpenAI` + `base_url`：

| 服务商 | 推荐用法 |
|---|---|
| OpenRouter | `pip install langchain-openrouter` → `ChatOpenRouter(model="openai/gpt-5.4-mini")` |
| LiteLLM | `pip install langchain-litellm` → `ChatLiteLLM` |

**一句话总结**：`response_metadata` 里能看到什么，取决于你用的是哪家 provider 的解析器，而不是接口返回了什么。OpenAI 的解析器不认识 OpenRouter 的字段，就当垃圾丢了。

---

### 小结：日常只关心这几个

| 你想干嘛 | 取哪个字段 |
|---|---|
| 拿到回复文本 | `result.content` |
| 看 token 花费 | `result.usage_metadata` |
| 看花了多少钱 | `result.response_metadata['token_usage']['cost']` |
| 看是否被截断 | `result.response_metadata['finish_reason']` |
| 处理工具调用 | `result.tool_calls` |
| 报 bug / 找客服 | `result.response_metadata['id']` |